# v08 - ConvNeXt-tiny 2.5D model on the v07 cache (GPU T4 x2)

Third step of the own-model leg (`docs/research/v06-own-model-design.md` revision 3.3,
decision D-006; Gate S runs on our own v07 cache, decision D-007).

- **Input:** per study, 5 slots; each token is a triplet (c-1, c, c+1) of consecutive
  stored slices of one slot as RGB, 320 px. Training uses a covering sampler with
  `K_train` centres per slot; evaluation uses `K_infer` evenly spaced centres per slot.
- **Model:** `convnext_tiny.fb_in22k_ft_in1k` (timm) per token, plus a slot embedding,
  gated per-finding attention pooling over all tokens of a study, 12 logits.
- **Targets:** soft targets and per-cell loss weights from v06 (UNK cells weighted 0.3);
  gold studies are never trained on.
- **Arms:** `K_train = 4` and `K_train = 8` run in parallel, one per T4, in separate
  processes (Gate A0).

Modes (`MODE` in the configuration cell):

- `auto` (decision D-008) - fold 0 with both arms, then the Gate A0 / Gate B decision,
  then, if Gate B passes and time remains, folds 1 and 2 in parallel with the chosen
  `K_train`. Everything stops cleanly before `GLOBAL_DEADLINE_S` so the run is saved
  before Kaggle's session limit. Each arm starts with a memory probe (below).

- `smoke` - **Gate S**: about 300 micro-steps per arm and one full validation pass; records
  peak memory, images/s, data-loading share and validation time, and projects the fold
  time. Pass: peak reserved memory < 14 GB, projected fold <= 4 h for `K_train = 4`
  (<= 5 h for 8).
- `fold` - trains `FOLD` for `EPOCHS`, keeps the best epoch by validation macro AUC
  (`K_infer = 8`, from epoch `EVAL_FROM_EPOCH`), then evaluates the best checkpoint with
  `K_infer` in {`K_train`, 8, 16, 24} (Gate A) and on the gold studies, and applies the
  Gate A0 rule.

Every arm stops cleanly at its time limit (design section 5.1), saving its state and a
receipt marked `stopped_early`. Before training, every arm runs a **memory probe**: one
forward/backward pass at its largest micro-batch; on out-of-memory it enables activation
checkpointing, and then halves the micro-batch, and records what it did.

Inputs: outputs of `lingxd/v06-dicom-audit` and `lingxd/v07-cache-320`; internet on for
the ImageNet weights (training only; submissions use attached weights).

In [ ]:
# v08 configuration
import json
import os
import subprocess
import sys
import time
from pathlib import Path

VERSION = 'v08'
MODE = 'auto'             # 'auto' (D-008), 'smoke' (Gate S) or 'fold'
FOLD = 0                  # fold for 'smoke' / 'fold' and stage 1 of 'auto'
STAGE2_FOLDS = [1, 2]     # 'auto': folds trained in parallel after stage 1
NOTEBOOK_T0 = time.time()
GLOBAL_DEADLINE_S = 11.0 * 3600   # stop everything before Kaggle's 12 h session limit
WORK = Path('/kaggle/working') / VERSION
WORK.mkdir(parents=True, exist_ok=True)

COMMON = {
    'version': VERSION,
    'mode': MODE,
    'fold': FOLD,
    'backbone': 'convnext_tiny.fb_in22k_ft_in1k',
    'pretrained': True,
    'img': 320,
    'epochs': 10,
    'lr_head': 2e-4,
    'lr_backbone': 5e-5,
    'weight_decay': 0.05,
    'warmup_epochs': 1,
    'effective_batch_studies': 8,
    'grad_checkpointing': False,
    'eval_from_epoch': 4,
    'k_select': 8,
    'k_infer_grid': [8, 16, 24],
    'smoke_steps': 300,
    'smoke_k_infer': 16,
    'eval_batch_tokens': 160,
    'num_workers': 2,
    'seed': 2026,
    'device': 'cuda',
}
ARMS = [
    {'name': 'k4', 'k_train': 4, 'micro_batch_studies': 2, 'gpu': 0},
    {'name': 'k8', 'k_train': 8, 'micro_batch_studies': 1, 'gpu': 1},
]
# Hard stop per arm (design 5.1); 'auto' further caps it by the global deadline.
TIME_LIMIT_S = 1.5 * 3600 if MODE == 'smoke' else 6.0 * 3600
GATE_B = {'macro_min': 0.86, 'meniscus_min': 0.80, 'label_min': 0.65}
GATE_S_MAX_RESERVED_GB = 14.0
GATE_S_MAX_FOLD_HOURS = {'k4': 4.0, 'k8': 5.0}
print(f'{VERSION}: mode={MODE}, fold={FOLD}, time limit per arm {TIME_LIMIT_S / 3600:.1f} h', flush=True)

In [ ]:
# Locate the v06 and v07 outputs without walking the competition image tree.
def find_unique_file(name, root=Path('/kaggle/input')):
    hits = []
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series', 'competitions', 'cache')
                   and 'rsna-knee-abnormality-detection' not in d]
        if name in files:
            hits.append(Path(current) / name)
    if len(hits) != 1:
        raise FileNotFoundError(f'expected one {name} under {root}, found {hits}')
    return hits[0]

PATHS = {
    'targets': str(find_unique_file('v06_targets.csv')),
    'folds': str(find_unique_file('v06_folds.csv')),
    'cache_studies': str(find_unique_file('v07_cache_studies.csv')),
}
PATHS['cache_dir'] = str(Path(PATHS['cache_studies']).parent / 'cache')
assert Path(PATHS['cache_dir']).is_dir(), PATHS['cache_dir']
print(json.dumps(PATHS, indent=2))

## Trainer script

Written to disk so that each arm runs in its own process on its own GPU. The notebook is
the source of truth; the script is regenerated on every run.

In [ ]:
%%writefile /kaggle/working/v08_train.py
"""v08 trainer: one arm (one K_train) on one GPU. Configuration comes as a JSON argument."""
import json
import math
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
FOCAL = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus']
N_SLOTS = 5
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)


# ----------------------------------------------------------------------------- data
def load_study(cache_dir, uid):
    with np.load(Path(cache_dir) / f'{uid}.npz') as z:
        return z['jpeg'], z['offsets'], z['slot']


def decode(jpeg, offsets, k):
    return cv2.imdecode(np.frombuffer(jpeg[offsets[k]:offsets[k + 1]].tobytes(), np.uint8),
                        cv2.IMREAD_GRAYSCALE)


def slot_centres(n, k, train, rng):
    if train:
        edges = np.linspace(0, n, k + 1)
        c = np.floor(edges[:-1] + rng.random(k) * (edges[1:] - edges[:-1])).astype(int)
    else:
        c = np.linspace(0, n - 1, k).round().astype(int)
    return np.clip(c, 0, n - 1)


def augment(tri, rng):
    # tri: (3, H, W) uint8 triplet; the same geometric transform for all three channels.
    h, w = tri.shape[1:]
    angle = rng.uniform(-10, 10)
    scale = rng.uniform(0.9, 1.1)
    tx, ty = rng.uniform(-0.05, 0.05) * w, rng.uniform(-0.05, 0.05) * h
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    m[:, 2] += (tx, ty)
    img = cv2.warpAffine(np.ascontiguousarray(tri.transpose(1, 2, 0)), m, (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    img = img.astype(np.float32) / 255.0
    gamma = rng.uniform(0.85, 1.15)
    img = np.clip(img, 0, 1) ** gamma
    img = img * rng.uniform(0.9, 1.1) + rng.uniform(-0.05, 0.05)
    return np.clip(img, 0, 1).transpose(2, 0, 1)


class StudyDataset(Dataset):
    def __init__(self, uids, targets, weights, cache_dir, k, train, img, seed):
        self.uids, self.targets, self.weights = list(uids), targets, weights
        self.cache_dir, self.k, self.train, self.img, self.seed = cache_dir, k, train, img, seed
        self.epoch = 0

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed, self.epoch, i)) if self.train else None
        jpeg, offsets, slot = load_study(self.cache_dir, self.uids[i])
        tokens, slots, cache = [], [], {}
        for s in range(N_SLOTS):
            idx = np.flatnonzero(slot == s)
            if len(idx) == 0:
                continue
            n = len(idx)
            for c in slot_centres(n, self.k, self.train, rng):
                chans = []
                for d in (-1, 0, 1):
                    j = int(idx[min(max(c + d, 0), n - 1)])
                    if j not in cache:
                        cache[j] = decode(jpeg, offsets, j)
                    chans.append(cache[j])
                tri = np.stack(chans)
                if tri.shape[1] != self.img:
                    tri = np.stack([cv2.resize(x, (self.img, self.img), interpolation=cv2.INTER_AREA) for x in tri])
                x = augment(tri, rng) if self.train else tri.astype(np.float32) / 255.0
                tokens.append((x - MEAN) / STD)
                slots.append(s)
        return {'x': np.stack(tokens).astype(np.float32), 'slot': np.asarray(slots, np.int64),
                'y': self.targets[i], 'w': self.weights[i], 'uid': self.uids[i]}


def collate(batch):
    return batch


# ---------------------------------------------------------------------------- model
class Net(nn.Module):
    def __init__(self, backbone, pretrained, n_out=12, att_dim=256, drop=0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        d = self.backbone.num_features
        self.slot_emb = nn.Embedding(N_SLOTS, d)
        self.norm = nn.LayerNorm(d)
        self.att_v = nn.Linear(d, att_dim)
        self.att_u = nn.Linear(d, att_dim)
        self.att_w = nn.Linear(att_dim, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.normal_(self.head_w, std=0.01)

    def encode(self, x, chunk=None):
        if chunk is None or x.shape[0] <= chunk:
            return self.backbone(x)
        return torch.cat([self.backbone(x[i:i + chunk]) for i in range(0, x.shape[0], chunk)])

    def pool(self, h, slot, counts):
        # h: (N, D) tokens of several studies concatenated; counts: tokens per study.
        h = self.norm(h + self.slot_emb(slot))
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).float()  # (N, 12)
        out, start = [], 0
        for n in counts:
            hs, as_ = h[start:start + n].float(), a[start:start + n]
            wts = torch.softmax(as_, dim=0)                                  # (n, 12)
            pooled = torch.einsum('nk,nd->kd', wts, hs)                      # (12, D)
            out.append((self.drop(pooled) * self.head_w).sum(-1) + self.head_b)
            start += n
        return torch.stack(out)                                              # (B, 12)


def batch_tensors(batch, device):
    x = torch.from_numpy(np.concatenate([b['x'] for b in batch])).to(device, non_blocking=True)
    slot = torch.from_numpy(np.concatenate([b['slot'] for b in batch])).to(device)
    counts = [len(b['slot']) for b in batch]
    return x, slot, counts


# ------------------------------------------------------------------------- metrics
def macro_auc(y_true_bin, y_score):
    per = {}
    for j, name in enumerate(LABELS):
        col = y_true_bin[:, j]
        if col.min() != col.max():
            per[name] = float(roc_auc_score(col, y_score[:, j]))
    return float(np.mean(list(per.values()))) if per else float('nan'), per


def bootstrap_ci(y_true_bin, y_score, n=200, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        s = rng.integers(0, len(y_true_bin), len(y_true_bin))
        m, _ = macro_auc(y_true_bin[s], y_score[s])
        if np.isfinite(m):
            vals.append(m)
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))] if vals else [None, None]


@torch.no_grad()
def predict(model, ds, device, batch_tokens, amp):
    model.eval()
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=CFG['num_workers'], collate_fn=collate)
    preds, uids = [], []
    for batch in loader:
        x, slot, counts = batch_tensors(batch, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            h = model.encode(x, chunk=batch_tokens)
            logits = model.pool(h, slot, counts)
        preds.append(torch.sigmoid(logits.float()).cpu().numpy())
        uids += [b['uid'] for b in batch]
    return uids, np.concatenate(preds)


# --------------------------------------------------------------------------- main
CFG = json.loads(sys.argv[1])


def main():
    t_start = time.time()
    arm, out = CFG['arm'], Path(CFG['out_dir'])
    out.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(CFG['seed'])
    device = torch.device(CFG['device'] if torch.cuda.is_available() or CFG['device'] == 'cpu' else 'cpu')
    amp = device.type == 'cuda'

    targets = pd.read_csv(CFG['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    folds = pd.read_csv(CFG['folds'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    cached = pd.read_csv(CFG['cache_studies'], dtype={'StudyInstanceUID': str})
    ok = set(cached[cached.error.fillna('') == ''].StudyInstanceUID)
    fold_of = folds.fold.reindex(targets.index)
    train_ids = [u for u in targets.index if fold_of[u] >= 0 and fold_of[u] != CFG['fold'] and u in ok]
    val_ids = [u for u in targets.index if fold_of[u] == CFG['fold'] and u in ok]
    gold_ids = [u for u in targets.index if bool(targets.loc[u, 'is_gold']) and u in ok]
    Y = targets[LABELS].to_numpy(np.float32)
    W = targets[[f'{c}__weight' for c in LABELS]].to_numpy(np.float32)
    pos = {u: i for i, u in enumerate(targets.index)}
    pick = lambda ids, m: m[[pos[u] for u in ids]]
    log(f'arm {arm}: train {len(train_ids)}, val {len(val_ids)}, gold {len(gold_ids)}, device {device}')

    k = CFG['k_train']
    train_ds = StudyDataset(train_ids, pick(train_ids, Y), pick(train_ids, W), CFG['cache_dir'], k, True, CFG['img'], CFG['seed'])
    mb = CFG['micro_batch_studies']
    accum = max(1, CFG['effective_batch_studies'] // mb)
    model = Net(CFG['backbone'], CFG['pretrained']).to(device)
    if CFG['grad_checkpointing']:
        model.backbone.set_grad_checkpointing(True)
    probe = {'grad_checkpointing': bool(CFG['grad_checkpointing']), 'micro_batch_studies': mb, 'attempts': []}
    if device.type == 'cuda':
        while True:
            n_tok = mb * N_SLOTS * k
            try:
                xx = torch.randn(n_tok, 3, CFG['img'], CFG['img'], device=device)
                ss = torch.arange(n_tok, device=device) % N_SLOTS
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    lo = model.pool(model.encode(xx), ss, [N_SLOTS * k] * mb)
                lo.float().sum().backward()
                model.zero_grad(set_to_none=True)
                probe['attempts'].append({'tokens': n_tok, 'ckpt': probe['grad_checkpointing'], 'ok': True,
                                          'peak_reserved_gb': round(torch.cuda.max_memory_reserved() / 1e9, 2)})
                del xx, ss, lo
                torch.cuda.empty_cache()
                break
            except torch.OutOfMemoryError:
                model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                probe['attempts'].append({'tokens': n_tok, 'ckpt': probe['grad_checkpointing'], 'ok': False})
                if not probe['grad_checkpointing']:
                    model.backbone.set_grad_checkpointing(True)
                    probe['grad_checkpointing'] = True
                elif mb > 1:
                    mb = mb // 2
                    probe['micro_batch_studies'] = mb
                else:
                    raise
        accum = max(1, CFG['effective_batch_studies'] // mb)
        log(f'arm {arm}: memory probe {probe}')
    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    opt = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': CFG['lr_backbone']},
                             {'params': head_params, 'lr': CFG['lr_head']}], weight_decay=CFG['weight_decay'])
    base_lrs = [CFG['lr_backbone'], CFG['lr_head']]
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    steps_per_epoch = math.ceil(len(train_ids) / (mb * accum))
    total_steps = steps_per_epoch * CFG['epochs']
    warm = steps_per_epoch * CFG['warmup_epochs']

    def set_lr(step):
        f = step / max(1, warm) if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))
        for g, lr in zip(opt.param_groups, base_lrs):
            g['lr'] = lr * f

    receipt = {'arm': arm, 'k_train': k, 'mode': CFG['mode'], 'fold': CFG['fold'], 'status': 'running',
               'memory_probe': probe,
               'n_train': len(train_ids), 'n_val': len(val_ids), 'n_gold': len(gold_ids),
               'micro_batch_studies': mb, 'accum': accum, 'tokens_per_study_train': N_SLOTS * k}
    write = lambda: (out / f'v08_receipt_{arm}.json').write_text(json.dumps(receipt, indent=2))

    def time_up():
        return time.time() - t_start > CFG['time_limit_s']

    def save_state(tag, epoch, step):
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                    'epoch': epoch, 'step': step, 'cfg': CFG}, out / f'v08_fold_{CFG["fold"]}_{arm}_{tag}.pt')

    def run_eval(ids, k_infer):
        ds = StudyDataset(ids, pick(ids, Y), pick(ids, W), CFG['cache_dir'], k_infer, False, CFG['img'], CFG['seed'])
        t = time.time()
        uids, p = predict(model, ds, device, CFG['eval_batch_tokens'], amp)
        return uids, p, time.time() - t

    def score(ids, p, gold=False):
        if gold:
            yb = targets.loc[ids, [f'{c}__gold' for c in LABELS]].to_numpy(np.float32)
        else:
            yb = (pick(ids, Y) >= 0.5).astype(np.float32)
        m, per = macro_auc(yb, p)
        return m, per, yb

    step, micro, best, best_epoch, stopped = 0, 0, -1.0, -1, False
    t_data = t_compute = 0.0
    n_images = 0
    torch.cuda.reset_peak_memory_stats() if amp else None
    max_epochs = 1 if CFG['mode'] == 'smoke' else CFG['epochs']
    epoch_seconds = []
    for epoch in range(max_epochs):
        train_ds.epoch = epoch
        loader = DataLoader(train_ds, batch_size=mb, shuffle=True, num_workers=CFG['num_workers'],
                            collate_fn=collate, drop_last=True, persistent_workers=False,
                            generator=torch.Generator().manual_seed(CFG['seed'] + epoch))
        model.train()
        opt.zero_grad(set_to_none=True)
        t0 = time.time()
        for batch in loader:
            t1 = time.time()
            t_data += t1 - t0
            x, slot, counts = batch_tensors(batch, device)
            y = torch.from_numpy(np.stack([b['y'] for b in batch])).to(device)
            w = torch.from_numpy(np.stack([b['w'] for b in batch])).to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model.pool(model.encode(x), slot, counts)
            loss = (F.binary_cross_entropy_with_logits(logits.float(), y, reduction='none') * w).sum() / w.sum()
            scaler.scale(loss / accum).backward()
            micro += 1
            n_images += x.shape[0]
            if micro % accum == 0:
                set_lr(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                step += 1
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            t_compute += t0 - t1
            if micro % 50 == 0:
                log(f'arm {arm} epoch {epoch} micro {micro} loss {loss.item():.4f} '
                    f'img/s {n_images / max(t_data + t_compute, 1e-9):.1f} data share {t_data / max(t_data + t_compute, 1e-9):.2f}')
            if CFG['mode'] == 'smoke' and micro >= CFG['smoke_steps']:
                break
            if time_up():
                stopped = True
                break
        epoch_seconds.append(round(time.time() - t_start, 1))
        receipt['elapsed_at_epoch_end'] = epoch_seconds
        if stopped:
            save_state('last', epoch, step)
            receipt.update(status='stopped_early', stopped_epoch=epoch, stopped_step=step)
            break
        if CFG['mode'] != 'smoke' and epoch + 1 >= CFG['eval_from_epoch']:
            uids, p, _ = run_eval(val_ids, CFG['k_select'])
            m, per, _ = score(uids, p)
            log(f'arm {arm} epoch {epoch} val macro AUC (K_infer={CFG["k_select"]}) {m:.4f}')
            receipt.setdefault('epoch_val_auc', {})[str(epoch)] = m
            if m > best:
                best, best_epoch = m, epoch
                torch.save({'model': model.state_dict(), 'epoch': epoch, 'cfg': CFG},
                           out / f'v08_fold_{CFG["fold"]}_{arm}_best.pt')
        write()

    elapsed = t_data + t_compute
    receipt.update(train_images=n_images, train_seconds=round(elapsed, 1),
                   train_images_per_s=round(n_images / max(elapsed, 1e-9), 1),
                   data_share=round(t_data / max(elapsed, 1e-9), 3), micro_steps=micro, optimizer_steps=step)
    if amp:
        receipt.update(peak_allocated_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                       peak_reserved_gb=round(torch.cuda.max_memory_reserved() / 1e9, 2))

    if CFG['mode'] == 'smoke':
        uids, p, val_s = run_eval(val_ids, CFG['smoke_k_infer'])
        m, _, _ = score(uids, p)
        sec_per_micro = elapsed / max(micro, 1)
        micro_per_epoch = len(train_ids) // mb
        val_k8_s = val_s * 8 / CFG['smoke_k_infer']
        n_evals = CFG['epochs'] - CFG['eval_from_epoch'] + 1
        final_grid_s = val_s * (k + 8 + 16 + 24) / CFG['smoke_k_infer']
        fold_s = sec_per_micro * micro_per_epoch * CFG['epochs'] + n_evals * val_k8_s + final_grid_s
        receipt.update(status='smoke_done', val_seconds_k16=round(val_s, 1), smoke_val_macro_auc=m,
                       projected_fold_hours=round(fold_s / 3600, 2), seconds_per_micro_step=round(sec_per_micro, 3))
        write()
        log(json.dumps(receipt))
        return

    if best_epoch < 0:
        write()
        log(json.dumps(receipt))
        return
    # Gate A and gold evaluation on the best checkpoint (also after an early stop).
    model.load_state_dict(torch.load(out / f'v08_fold_{CFG["fold"]}_{arm}_best.pt', map_location=device)['model'])
    grid = sorted(set([k] + CFG['k_infer_grid']))
    gate_a = {}
    for k_inf in grid:
        uids, p, _ = run_eval(val_ids, k_inf)
        m, per, _ = score(uids, p)
        gate_a[str(k_inf)] = {'macro': m, 'per_label': per}
        pd.DataFrame(p, columns=LABELS, index=pd.Index(uids, name='StudyInstanceUID')).to_csv(
            out / f'v08_oof_fold{CFG["fold"]}_{arm}_k{k_inf}.csv')
    uids, p, _ = run_eval(gold_ids, 16)
    gm, gper, gyb = score(uids, p, gold=True)
    pd.DataFrame(p, columns=LABELS, index=pd.Index(uids, name='StudyInstanceUID')).to_csv(
        out / f'v08_gold_fold{CFG["fold"]}_{arm}_k16.csv')
    receipt.update(status='done' if receipt['status'] != 'stopped_early' else 'stopped_early_evaluated',
                   best_epoch=best_epoch, best_val_macro_k8=best, gate_a=gate_a,
                   gold_macro_k16=gm, gold_ci=bootstrap_ci(gyb, p), gold_per_label=gper,
                   total_seconds=round(time.time() - t_start, 1))
    write()
    log(json.dumps({k_: v for k_, v in receipt.items() if k_ != 'gate_a'}))


if __name__ == '__main__':
    main()

## Launch the arms (one process per GPU)

In [ ]:
def remaining_s():
    return GLOBAL_DEADLINE_S - (time.time() - NOTEBOOK_T0)

def run_arms(specs, time_limit_s):
    # specs: list of dicts with name, k_train, micro_batch_studies, gpu, fold, mode.
    procs = []
    for s in specs:
        tag = f'fold{s["fold"]}_{s["name"]}'
        cfg = {**COMMON, **PATHS, 'mode': s['mode'], 'fold': s['fold'], 'arm': s['name'],
               'k_train': s['k_train'], 'micro_batch_studies': s['micro_batch_studies'],
               'time_limit_s': time_limit_s, 'out_dir': str(WORK / tag)}
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(s['gpu'])}
        logf = open(WORK / f'v08_log_{tag}.txt', 'w')
        procs.append((tag, subprocess.Popen([sys.executable, '/kaggle/working/v08_train.py', json.dumps(cfg)],
                                            env=env, stdout=logf, stderr=subprocess.STDOUT), logf))
        print(f'launched {tag} on GPU {s["gpu"]}, time limit {time_limit_s / 3600:.2f} h', flush=True)
    out = {}
    for tag, p, logf in procs:
        p.wait()
        logf.close()
        print(tag, 'exit code', p.returncode, flush=True)
        print(''.join(open(WORK / f'v08_log_{tag}.txt').readlines()[-12:]), flush=True)
        name = tag.split('_', 1)[1]
        f = WORK / tag / f'v08_receipt_{name}.json'
        out[tag] = json.loads(f.read_text()) if f.is_file() else {'status': 'missing', 'returncode': p.returncode}
    return out

stage1_mode = 'smoke' if MODE == 'smoke' else ('fold' if MODE == 'fold' else 'auto')
stage1 = run_arms([{**a, 'fold': FOLD, 'mode': stage1_mode} for a in ARMS],
                  min(TIME_LIMIT_S, remaining_s() - 15 * 60))

## Gate S / Gate A0 decision

In [ ]:
import numpy as np
receipts = {a['name']: stage1[f'fold{FOLD}_{a["name"]}'] for a in ARMS}
decision = {'mode': MODE, 'fold': FOLD}
if MODE == 'smoke':
    for name, r in receipts.items():
        ok = (r.get('status') == 'smoke_done' and r.get('peak_reserved_gb', 99) < GATE_S_MAX_RESERVED_GB
              and r.get('projected_fold_hours', 99) <= GATE_S_MAX_FOLD_HOURS[name])
        decision[f'gate_s_{name}'] = {'pass': bool(ok), **{k: r.get(k) for k in (
            'status', 'peak_reserved_gb', 'train_images_per_s', 'data_share', 'val_seconds_k16',
            'projected_fold_hours')}}
else:
    r4, r8 = receipts['k4'], receipts['k8']
    comparable = r4.get('status') == 'done' and r8.get('status') == 'done'
    decision['comparable'] = comparable
    usable = {n: r for n, r in receipts.items() if r.get('gate_a')}
    if not comparable:
        # A0 outcome table (design 4.5): prefer 4 if it finished; 8 only if 4 failed.
        if 'k4' in usable:
            decision['chosen_k_train'] = 4
        elif 'k8' in usable:
            decision['chosen_k_train'] = 8
    if comparable:
        a4, a8 = r4['gate_a']['16'], r8['gate_a']['16']
        d_macro = a8['macro'] - a4['macro']
        d_focal = {c: a8['per_label'].get(c, np.nan) - a4['per_label'].get(c, np.nan) for c in
                   ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus']}
        choose8 = d_macro >= 0.003 or (d_macro >= 0 and any(v >= 0.01 for v in d_focal.values()))
        decision.update(delta_macro_k16=d_macro, delta_focal_k16=d_focal, chosen_k_train=8 if choose8 else 4)
    else:
        decision['note'] = 'not comparable: apply the A0 outcome table in the design note'
    for name, r in receipts.items():
        decision[name] = {k: r.get(k) for k in ('status', 'best_epoch', 'best_val_macro_k8', 'gold_macro_k16',
                                                'gold_ci', 'total_seconds')}
        if r.get('gate_a'):
            decision[name]['gate_a_macro'] = {k: v['macro'] for k, v in r['gate_a'].items()}
if MODE != 'smoke' and decision.get('chosen_k_train'):
    r = receipts[f'k{decision["chosen_k_train"]}']['gate_a']['16']
    per = r['per_label']
    gate_b = (r['macro'] >= GATE_B['macro_min']
              and min(per.get('Medial Meniscus', 0), per.get('Lateral Meniscus', 0)) >= GATE_B['meniscus_min']
              and min(per.values()) >= GATE_B['label_min'])
    decision['gate_b'] = {'pass': bool(gate_b), 'macro_k16': r['macro'], 'min_label': min(per.values()),
                          'meniscus': [per.get('Medial Meniscus'), per.get('Lateral Meniscus')]}
print(json.dumps(decision, indent=2, default=float))
(WORK / f'v08_decision_{MODE}.json').write_text(json.dumps(decision, indent=2, default=float))

## Stage 2 (`auto` only): folds 1 and 2 with the chosen `K_train`

Runs only if Gate B passed and the chosen arm's measured fold-0 duration, times 1.15,
fits in the time left before the global deadline.

In [ ]:
stage2 = {}
if MODE == 'auto':
    k_sel = decision.get('chosen_k_train')
    chosen = next((a for a in ARMS if a['k_train'] == k_sel), None)
    fold0_s = receipts.get(f'k{k_sel}', {}).get('total_seconds') if k_sel else None
    left = remaining_s() - 15 * 60
    go = bool(chosen and decision.get('gate_b', {}).get('pass') and fold0_s and 1.15 * fold0_s <= left)
    decision['stage2'] = {'launched': go, 'fold0_seconds': fold0_s, 'seconds_left': round(left)}
    print(json.dumps(decision['stage2']), flush=True)
    if go:
        specs = [{**chosen, 'gpu': g, 'fold': f, 'mode': 'auto'} for g, f in enumerate(STAGE2_FOLDS)]
        stage2 = run_arms(specs, left)
        decision['stage2']['results'] = {tag: {k: r.get(k) for k in ('status', 'best_epoch', 'best_val_macro_k8',
                                                                      'gold_macro_k16', 'total_seconds')}
                                         for tag, r in stage2.items()}
    (WORK / f'v08_decision_{MODE}.json').write_text(json.dumps(decision, indent=2, default=float))
print(f'notebook elapsed {(time.time() - NOTEBOOK_T0) / 3600:.2f} h', flush=True)